# NASA CMAPSS — Data Pipeline
## Combine all 4 datasets into unified train.csv and test.csv

This notebook:
1. Loads all 4 training files (FD001–FD004) and combines them
2. Loads all 4 test files + RUL ground truth and combines them
3. Assigns globally unique `engine_id` across datasets
4. Computes RUL (Remaining Useful Life) for all rows
5. Caps RUL at 125 cycles
6. Saves `data/train.csv` and `data/test.csv`

In [ ]:
import pandas as pd
import numpy as np

# Column names: 2 IDs + 3 operational settings + 21 sensors = 26 columns
cols = ['engine_id', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

RUL_CAP = 125  # Cap RUL at 125 cycles (standard practice)

## Load & Combine Training Data

For each FD00X file:
- Load the raw text file
- Add `dataset_id` (1–4) to track which subset it came from
- Offset `engine_id` so IDs are globally unique across all 4 files
- Compute RUL = `max_cycle_for_engine - current_cycle`
- Cap RUL at 125

In [ ]:
train_dfs = []
engine_id_offset = 0

for i in range(1, 5):
    df = pd.read_csv(f'data/train_FD00{i}.txt', sep=r'\s+', header=None, names=cols)
    df['dataset_id'] = i

    # Make engine_id globally unique across all 4 datasets
    df['engine_id'] = df['engine_id'] + engine_id_offset
    engine_id_offset = df['engine_id'].max()

    # Compute RUL: for training data, last cycle = failure
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    df = df.merge(max_cycles, on='engine_id')
    df['RUL'] = df['max_cycle'] - df['cycle']
    df.drop('max_cycle', axis=1, inplace=True)

    train_dfs.append(df)
    print(f"FD00{i}: {df['engine_id'].nunique()} engines, {len(df)} rows, "
          f"engine_id range: {df['engine_id'].min()}-{df['engine_id'].max()}")

train = pd.concat(train_dfs, ignore_index=True)

# Cap RUL at 125
train['RUL'] = train['RUL'].clip(upper=RUL_CAP)

print(f"\n--- Combined Training Data ---")
print(f"Shape: {train.shape}")
print(f"Total engines: {train['engine_id'].nunique()}")
print(f"RUL range: {train['RUL'].min()} to {train['RUL'].max()}")
train.head()

## Load & Combine Test Data + RUL Ground Truth

For test data, the time series is truncated before failure. The `RUL_FD00X.txt` file provides the true RUL at each engine's **last observed cycle**.

We compute RUL for every row in the test set:
- `RUL_at_row = RUL_at_last_cycle + (max_cycle - current_cycle)`

In [ ]:
test_dfs = []
engine_id_offset = 0

for i in range(1, 5):
    df = pd.read_csv(f'data/test_FD00{i}.txt', sep=r'\s+', header=None, names=cols)
    rul = pd.read_csv(f'data/RUL_FD00{i}.txt', header=None, names=['RUL'])

    df['dataset_id'] = i

    # Make engine_id globally unique
    df['engine_id'] = df['engine_id'] + engine_id_offset
    engine_id_offset = df['engine_id'].max()

    # Map RUL to each engine (RUL file has one value per engine, in order)
    unique_engines = sorted(df['engine_id'].unique())
    rul_map = pd.DataFrame({'engine_id': unique_engines, 'rul_last': rul['RUL'].values})

    # Compute RUL for every row:
    # RUL_at_row = RUL_at_last_cycle + (last_cycle - current_cycle)
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    df = df.merge(max_cycles, on='engine_id')
    df = df.merge(rul_map, on='engine_id')
    df['RUL'] = df['rul_last'] + (df['max_cycle'] - df['cycle'])
    df.drop(['max_cycle', 'rul_last'], axis=1, inplace=True)

    test_dfs.append(df)
    print(f"FD00{i}: {df['engine_id'].nunique()} engines, {len(df)} rows, "
          f"engine_id range: {df['engine_id'].min()}-{df['engine_id'].max()}")

test = pd.concat(test_dfs, ignore_index=True)

# Cap RUL at 125 for consistency
test['RUL'] = test['RUL'].clip(upper=RUL_CAP)

print(f"\n--- Combined Test Data ---")
print(f"Shape: {test.shape}")
print(f"Total engines: {test['engine_id'].nunique()}")
print(f"RUL range: {test['RUL'].min()} to {test['RUL'].max()}")
test.head()

## Save to CSV

In [ ]:
train.to_csv('data/raw/train.csv', index=False)
test.to_csv('data/raw/test.csv', index=False)
print("Saved: data/raw/train.csv and data/raw/test.csv")

## Verification

In [ ]:
print("=== VERIFICATION ===\n")

# Check engine counts
print(f"Train engines: {train['engine_id'].nunique()} (expected: 100+260+100+248 = 708)")
print(f"Test engines:  {test['engine_id'].nunique()} (expected: 100+259+100+249 = 708)")

# Check RUL range
print(f"\nTrain RUL range: {train['RUL'].min()} to {train['RUL'].max()} (should be 0 to {RUL_CAP})")
print(f"Test RUL range:  {test['RUL'].min()} to {test['RUL'].max()}")

# Check that last row of each training engine has RUL = 0
last_rows = train.groupby('engine_id')['RUL'].last()
assert (last_rows == 0).all(), "ERROR: Not all training engines end with RUL=0!"
print("\nAll training engines end with RUL=0")

# Check dataset distribution
print(f"\nDataset distribution (train):")
print(train.groupby('dataset_id')['engine_id'].nunique().to_string())

print(f"\nDataset distribution (test):")
print(test.groupby('dataset_id')['engine_id'].nunique().to_string())

# Column check
print(f"\nColumns: {list(train.columns)}")
print(f"\nTrain shape: {train.shape}")
print(f"Test shape:  {test.shape}")

In [ ]:
print("\nData loading and verification complete!")